# M5 로지스틱 회귀 — 실습 (W7)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. **뉴런 하나 손계산**(z=2 → σ(2)=0.881)과 **벌점표**(MSE 천장 vs −log 벼랑)를 코드로 재현한다 ⭐
2. 유방암(1=악성, M2b 그대로)에서 표준화 파이프라인으로 **놓친 환자 9 → 4명**을 확인한다
3. **임계값 손잡이**(0.3/0.5/0.7 → FN 2/4/5)와 직선 결정경계를 확인한다

**7단계 멘탈모델 초점:** 모델 + 손실

## Part A. 세트피스 1 — 세상에서 가장 작은 로지스틱 ⭐
시그모이드 다섯 점(대칭!)과 뉴런 손계산(x=(3,1), w=(1,−2), b=1 → z=2 → 0.881)을 먼저 종이에서 완주하고 코드로 검산하세요.

In [ ]:
import numpy as np                                     # 수치 계산
import matplotlib.pyplot as plt                        # 그래프

def sigmoid(z):                                        # S자 곡선
    return 1 / (1 + np.exp(___))                       # ✍️ 빈칸: 지수 부분(마이너스 제트)

for zv in (-4, -2, 0, 2, 4):                           # 다섯 점 손계산 검산
    print(f'σ({zv:>2}) = {sigmoid(zv):.3f}')            # 0.018 / 0.119 / 0.5 / 0.881 / 0.982

x = np.array([3.0, 1.0])                               # 입력
w = np.array([1.0, -2.0])                              # 가중치
z = w @ x + ___                                        # ✍️ 빈칸: 절편 b의 값(세트피스의 그 수)
print('z =', z, '→ σ(z) =', round(sigmoid(z), 4))       # 2.0 → 0.8808
print('판정:', '클래스 1' if sigmoid(z) >= 0.5 else '클래스 0', '(88% 확률)')

zs = np.linspace(-8, 8, 200)                           # 곡선 전체도 그려 보기
plt.plot(zs, sigmoid(zs))
plt.axhline(0.5, color='r', linestyle='--', label='0.5 threshold')
plt.axvline(0, color='gray', linestyle=':')            # z=0 = 경계
plt.xlabel('z = w.x + b'); plt.ylabel('probability')   # 축(영어)
plt.title('Sigmoid'); plt.legend(); plt.show()

> **검산 포인트:** 다섯 점 0.018/0.119/**0.5**/0.881/0.982(대칭). 뉴런 완주: z = 1×3 + (−2)×1 + 1 = **2** → σ(2) = **0.881** → 클래스 1. σ(z)≥0.5 ⇔ **z≥0** → 경계는 z=0인 **직선**. 이 구조(가중합→S자→출력) = **뉴런 하나**(M9·2학기 D1의 복선).

## Part B. 세트피스 2 — 손실 교체극: MSE 천장 vs −log 벼랑 ⭐
정답이 1일 때 예측 확률 p의 벌점을 두 자로 비교합니다. 확신에 찬 오답(p=0.01)에서 무슨 일이?

In [ ]:
ps = np.array([0.9, 0.5, 0.1, 0.01])                   # 예측 확률 네 가지
pen_mse = (1 - ps) ** 2                                # MSE 벌점(정답 1)
pen_ce = -np.___(ps)                                   # ✍️ 빈칸: 교차 엔트로피 벌점 = 마이너스 (자연)로그
for p, m, c in zip(ps, pen_mse, pen_ce):
    print(f'p={p:>4}: MSE {m:.3f} | -log {c:.3f}')      # 0.98 천장 vs 4.605 벼랑!

pp = np.linspace(0.001, 0.999, 300)                    # 곡선 비교
plt.plot(pp, (1 - pp) ** 2, label='MSE penalty (ceiling <= 1)')
plt.plot(pp, -np.log(pp), label='-log penalty (cliff!)')
plt.xlabel('predicted probability p (true label = 1)') # 축(영어)
plt.ylabel('penalty')
plt.legend(); plt.grid(True, alpha=0.3)
plt.title('Why cross-entropy: punishing confident mistakes')
plt.show()

> **관찰:** p=0.01(확신에 찬 오답)에서 MSE는 **0.98(천장 ≤1)**, −log는 **4.605(벼랑 — p→0이면 ∞)**. "그렇게 자신 있게 틀렸어?"를 벌하는 자 = **교차 엔트로피**. 학습 문법은 M4 그대로(손실을 경사하강으로) — 자만 교체. (2학기 D1의 MSE·CE 손계산 대조에서 재회.)

## Part C. 실전 — 유방암, M2b와의 재회 (FN 9 → 4)
M2b와 **같은 데이터·같은 분할·같은 라벨**(`y = 1 − target`, 1=악성). M2b의 로지스틱(스케일링 없음)은 놓친 환자 9명이었습니다 — 표준화 파이프라인을 붙이면?

In [ ]:
from sklearn.datasets import load_breast_cancer        # 유방암 데이터
from sklearn.model_selection import train_test_split   # 분할(M2a)
from sklearn.linear_model import LogisticRegression    # 로지스틱 회귀
from sklearn.preprocessing import StandardScaler       # 표준화(M3)
from sklearn.pipeline import make_pipeline             # 누수 없는 배관(M3)
from sklearn.metrics import confusion_matrix, recall_score  # 성적표(M2b)

data = load_breast_cancer()                            # 569명, 특징 30개
X = data.data
y = ___ - data.target                                  # ✍️ 빈칸: 라벨 뒤집기 — 1=악성으로(M2b의 그 함정!)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = make_pipeline(StandardScaler(),                # M3의 교훈을 앞에
                      LogisticRegression(max_iter=5000))
model.fit(Xtr, ytr)                                    # 학습
print('정확도:', round(model.score(Xte, yte), 3))       # 0.971
y_pred = model.predict(Xte)                            # 기본(0.5) 판정
print(confusion_matrix(yte, y_pred))                   # [[106 1] [4 60]] — M2b는 [[106 1] [9 55]]!
print('재현율:', round(recall_score(yte, y_pred), 3),   # 0.938 (M2b는 0.859)
      '| 놓친 환자(FN):', int(((yte == 1) & (y_pred == 0)).sum()))  # 4명 — 9명에서!

> **관찰:** 모델은 M2b와 같은 로지스틱 — **표준화 하나로 놓친 환자 9 → 4명**(재현율 0.859 → 0.938). M3의 원칙("거리·수렴 계산은 발언권 통일 후에")이 **사람 수**로 돌아온 순간. 다음: 문턱을 조절하면?

## Part D. 임계값 손잡이 — FN 2/4/5
`predict_proba`로 확률을 꺼내 임계값 0.3/0.5/0.7의 성적을 비교합니다. 암 검진에선 어느 쪽으로 돌려야 할까요?

In [ ]:
from sklearn.metrics import precision_score            # 정밀도(M2b)

proba = model.predict_proba(___)[:, 1]                 # ✍️ 빈칸: 시험 데이터의 악성(1) 확률
print('앞 5명의 악성 확률:', np.round(proba[:5], 3))     # 확률이라는 보너스

for th in (0.3, 0.5, 0.7):                             # 문턱 세 가지
    p = (proba >= ___).astype(int)                     # ✍️ 빈칸: 이번 반복의 임계값으로 판정
    fn = int(((yte == 1) & (p == 0)).sum())            # 놓친 환자
    fp = int(((yte == 0) & (p == 1)).sum())            # 오경보
    print(f'th={th}: 정밀도 {precision_score(yte, p):.3f} | 재현율 {recall_score(yte, p):.3f}'
          f' | 놓침 FN={fn} | 오경보 FP={fp}')

> **관찰:** 0.3 → **FN 2**(재현율 0.969)·FP 1 / 0.5 → FN 4 / 0.7 → FN 5·FP 0(정밀도 1.0). **임계값 = 업무 비용으로 정하는 손잡이** — 놓침이 치명적(암 검진)이면 ↓, 오경보가 비싸면 ↑. M2b의 정밀도↔재현율 상충을 손잡이로 조절하는 실전. 최종 스코어: **놓친 9명(M2b) → 4명(스케일링) → 2명(임계값)**.

## Part E. 결정경계 — 직선임을 눈으로
2개 특징만 골라 경계를 그립니다. KNN(M3)의 구불구불함과 대조하세요.

In [ ]:
from matplotlib.colors import ListedColormap           # 색 지정

X2 = StandardScaler().fit_transform(X[:, [0, 1]])      # mean radius, mean texture(시각화용)
clf = LogisticRegression().fit(X2, ___)                # ✍️ 빈칸: 정답 라벨(1=악성으로 뒤집은 것)

xx, yy = np.meshgrid(np.linspace(X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5, 300),
                     np.linspace(X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5, 300))
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)  # 격자 전부 판정

plt.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#93c5fd', '#fca5a5']))
plt.scatter(X2[:, 0], X2[:, 1], c=y, edgecolor='k', s=15,
            cmap=ListedColormap(['#2563eb', '#dc2626']))
plt.xlabel('mean radius (scaled)'); plt.ylabel('mean texture (scaled)')  # 축(영어)
plt.title('Logistic regression: a straight-line boundary')
plt.show()

> **관찰:** 경계가 **직선** — z = w·x+b = 0인 곳(σ=0.5인 곳). KNN(M3)의 구불구불한 경계와 정반대. 직선으로 절대 못 가르는 데이터(XOR)는 **W14(M9)** 에서 만납니다 — 그때 이 뉴런을 "쌓는" 이유가 나옵니다.

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "x=(1,2), w=(2,−1), b=−1로 뉴런 손계산을 완주할 테니 채점해 줘."
- "MSE가 분류 손실로 부족한 이유를 벌점표(천장 vs 벼랑)로 설명해 볼게."
- "FN 9→4가 스케일링 덕분인 이유를 M3의 언어로 설명해 볼게."
- "스팸 필터라면 임계값을 올릴지 내릴지, 비용으로 논증해 볼게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 뉴런 하나(z=2 → σ(2)=0.881)와 벌점표(MSE 0.98 천장 vs −log 4.605 벼랑)를 손계산·코드로 완주했다
2. 유방암(1=악성)에서 표준화 파이프라인으로 **놓친 환자 9 → 4명**(재현율 0.859 → 0.938)을 확인했다
3. 임계값 손잡이(0.3/0.5/0.7 → FN 2/4/5)와 직선 경계를 확인했다

**스스로 점검**
- [ ] 시그모이드 다섯 점을 쓸 수 있다
- [ ] 뉴런 손계산을 새 숫자로 완주할 수 있다
- [ ] MSE 천장 vs CE 벼랑을 수치로 인용할 수 있다
- [ ] FN 9→4→2의 각 단계에서 무엇을 바꿨는지 안다
- [ ] 임계값을 어느 방향으로 돌릴지 업무 비용으로 논증할 수 있다

**🔹심화 (선택)**
- `C`(규제의 역수)를 0.01/1/100으로 바꿔 성능·계수 크기를 비교해 보세요.
- 정답이 0일 때의 벌점 −log(1−p)도 그려 보세요 — 대칭이 보입니다.
- `decision_function`(=z 값)을 꺼내 proba와 σ 관계를 검산해 보세요.

**다음 시간(M6):** 예/아니오 질문으로 공간을 자르는 결정트리 — 화이트박스.